In [ ]:
%matplotlib widget 

import numpy as np
import torch
import matplotlib.pyplot as plt
import pypulseq as pp
import MRzeroCore as mr0
import torchkbnufft as tkbn
import sys
sys.path.insert(0, "..")
from Modules import load_brain_phantom, default_system

torch.set_default_device("cuda:3")

In [ ]:
def truefisp_seq(
    fov: float = 250e-3,
    Nx: int = 64,
    alpha: float = 50.0,
    slice_thickness: float = 5e-3,
    TE: float = 3.98e-3,
    N_readout: int = 200,
    N_spokes: int = 32,
    N_dummy: int = 20,
    TI_start: float = 20e-3,
    TI_inter: float = 0.0,     
    TD: float = 2000e-3,
    write_seq: bool = False,
    plot: bool = False,
) -> pp.Sequence:

    system = default_system()
    seq = pp.Sequence(system)
    deltak = 1 / fov

    rf_inv, gz_inv, _ = pp.make_sinc_pulse(
        flip_angle=np.pi, duration=10e-3, slice_thickness=slice_thickness,
        apodization=0.5, time_bw_product=4, system=system,
        return_gz=True, use='inversion')

    rf, gz, _ = pp.make_sinc_pulse(
        apodization=0.5, duration=2e-3, flip_angle=alpha * np.pi / 180,
        slice_thickness=slice_thickness, system=system,
        time_bw_product=4, return_gz=True, use='excitation')

    gx_base = pp.make_trapezoid(channel='x', flat_area=Nx * deltak,
                                flat_time=6.4e-3 / 5, system=system)
    adc = pp.make_adc(num_samples=Nx, duration=gx_base.flat_time,
                      delay=gx_base.rise_time, system=system)

    gx_pre_base = pp.make_trapezoid(channel='x', area=-gx_base.area / 2,
                                    duration=2e-3, system=system)
    gz_reph     = pp.make_trapezoid(channel='z', area=-gz.area / 2,
                                    duration=2e-3, system=system)

    delay_TE = (
        np.ceil((
            TE
            - pp.calc_duration(gx_pre_base)
            - gz.fall_time - gz.flat_time / 2
            - pp.calc_duration(gx_base) / 2
        ) / seq.grad_raster_time) * seq.grad_raster_time
    )
    inv_duration   = pp.calc_duration(rf_inv, gz_inv)
    delay_TI_start = (
        np.ceil((TI_start - inv_duration / 2 - pp.calc_duration(gz) / 2)
                / seq.grad_raster_time) * seq.grad_raster_time
    )
    assert delay_TI_start > 0, 'TI_start too short'

    gre_block_dur = (
        pp.calc_duration(gx_pre_base)
        + pp.calc_duration(gz)          
        + delay_TE
        + pp.calc_duration(gx_base)     # readout
        + pp.calc_duration(gx_pre_base) # in-plane rewind
    )
    delay_TD = np.ceil(TD / seq.grad_raster_time) * seq.grad_raster_time
    delay_TI_inter = np.ceil(TI_inter / seq.grad_raster_time) * seq.grad_raster_time

    GA_DEG = 111.246
    #rf_phase = 0.0
    #rf_inc = 180.0          # FIX 4: alternate RF/ADC phase by 180 deg every TR
    spoke_counter = 0

    for seg in range(N_spokes):
        seq.add_block(rf_inv, gz_inv)
        seq.add_block(pp.make_delay(delay_TI_start))

        n_this_seg = N_readout + (N_dummy if seg == 0 else 0)
        for s in range(n_this_seg):
            is_dummy = (seg == 0) and (s < N_dummy)

            #rf.phase_offset  = rf_phase / 180 * np.pi
            #adc.phase_offset = rf_phase / 180 * np.pi
            #rf_inc   = rf_inc % 360
            #rf_phase = (rf_phase + rf_inc) % 360

            if is_dummy:
                angle = 0.0
            else:
                angle = np.deg2rad(spoke_counter * GA_DEG)
                spoke_counter += 1

            kx = np.cos(angle)
            ky = np.sin(angle)

            gx_pre = pp.make_trapezoid(channel='x', area=gx_pre_base.area * kx,
                                        duration=2e-3, system=system)
            gy_pre = pp.make_trapezoid(channel='y', area=gx_pre_base.area * ky,
                                        duration=2e-3, system=system)

            gx_rew = pp.make_trapezoid(channel='x', area=gx_pre_base.area * kx,   
                            duration=2e-3, system=system)
            gy_rew = pp.make_trapezoid(channel='y', area=gx_pre_base.area * ky,
                            duration=2e-3, system=system)

            #seq.add_block(gz_reph)
            seq.add_block(rf, gz)
            seq.add_block(gx_pre, gy_pre, gz_reph)
            seq.add_block(pp.make_delay(delay_TE))

            gx = pp.make_trapezoid(channel='x',
                                   amplitude=gx_base.amplitude * kx,
                                   flat_time=gx_base.flat_time,
                                   rise_time=gx_base.rise_time,
                                   fall_time=gx_base.fall_time,
                                   system=system)
            gy = pp.make_trapezoid(channel='y',
                                   amplitude=gx_base.amplitude * ky,
                                   flat_time=gx_base.flat_time,
                                   rise_time=gx_base.rise_time,
                                   fall_time=gx_base.fall_time,
                                   system=system)

            if is_dummy:
                seq.add_block(gx, gy)
            else:
                seq.add_block(gx, gy, adc)

            seq.add_block(gx_rew, gy_rew, gz_reph)

            if delay_TI_inter > 0 and s < n_this_seg - 1:
                seq.add_block(pp.make_delay(delay_TI_inter))

        seq.add_block(pp.make_delay(TD))

    ok, err = seq.check_timing()
    if ok:
        print('2-D TrueFISP timing check passed')
        print(f'  GRE block dur (TR) : {gre_block_dur * 1000:.2f} ms')
        print(f'  Readout train      : {N_readout * gre_block_dur * 1000:.0f} ms')
        est = N_spokes * (inv_duration + delay_TI_start
                          + N_readout * gre_block_dur + TD)
        print(f'  Scan time est       : {est / 60:.1f} min')
    else:
        print(f'Timing FAILED: {err}')

    seq.set_definition('FOV',  [fov, fov, slice_thickness])
    seq.set_definition('Name', 'trueFISP_2d_radial')
    if write_seq:
        seq.write('trueFISP_2D_Radial.seq')
    if plot:
        seq.plot(time_range=(0.2, 0.225))
    return seq, gre_block_dur, TI_start


seq, gre_block_dur, TI_start = truefisp_seq(write_seq=True, plot=True)

In [ ]:
device = torch.device("cuda:3")
N_readout = 200; N_spokes = 32; Nx = 64
Nx_img, Ny_img = Nx, Nx        
fov = 250e-3; deltak = 1 / fov
phantom = mr0.util.load_phantom()
phantom.B0 = 0 * phantom.B0
phantom.B1 = torch.ones(phantom.B1.shape, dtype=phantom.B1.dtype)
data = phantom.build()
mr0_seq = mr0.Sequence.import_file('trueFISP_2D_Radial.seq')
#signal_raw, _ = mr0.util.simulate('trueFISP_2D_Radial.seq', None)
graph = mr0.compute_graph(mr0_seq, data, max_state_count=150, min_state_mag=1e-4)
signal = mr0.execute_graph(graph, mr0_seq, data, print_progress=True)

In [ ]:
signal_all = signal.reshape(N_spokes, N_readout, Nx)
N_spokes = 32
N_readout = 200
Nx = 64

def reconstruct_readouts(
    signal_all, seq_file, readout_idx,
    N_readout, N_spokes, Nx_img, Ny_img, fov, device):

    deltak = 1 / fov
    readout_idx = list(readout_idx)
    assert all(0 <= n < N_readout for n in readout_idx)

    mr_seq     = mr0.Sequence.import_file(seq_file)
    ktraj_full = torch.tensor(mr_seq.get_kspace(), dtype=torch.float32)
    ktraj_full = ktraj_full.reshape(-1, ktraj_full.shape[-1])[:, :2]

    sig_list, ktraj_list = [], []
    for seg in range(N_spokes):
        for n in readout_idx:
            start = (seg * N_readout + n) * Nx_img
            ktraj_list.append(ktraj_full[start : start + Nx_img])
            sig_list.append(signal_all[seg, n, :])   # (Nx,) — single coil

    sig   = torch.cat(sig_list, dim=0).to(torch.complex64)
    ktraj = torch.cat(ktraj_list, dim=0)
    ktraj = ktraj / (deltak * Nx_img / 2) * np.pi * 0.85
    ktraj = ktraj.clamp(-np.pi, np.pi)
    ktraj = ktraj.T.contiguous().unsqueeze(0).to(device)

    r     = torch.linalg.norm(ktraj[0], dim=0)
    dcomp = r.view(1, 1, -1)

    sig      = sig.view(1, 1, -1).to(device)
    nufft_ob = tkbn.KbNufftAdjoint(
        im_size=(Nx_img, Ny_img), grid_size=(2 * Nx_img, 2 * Ny_img)
    ).to(device)
    image = nufft_ob(sig * dcomp, ktraj)[0, 0]
    return torch.abs(image).cpu().numpy()

def ti_eff(readout_idx, TI_start, gre_block_dur, TI_inter=0.0):
    centre = np.mean(list(readout_idx))
    return TI_start + centre * (gre_block_dur + TI_inter)

In [ ]:
Nx_img, Ny_img = Nx, Nx
fov = 250e-3; deltak = 1 / fov
readout_range = range(190, 200)

images, ti_effs = [], []
for n in readout_range:
    img = reconstruct_readouts(
        signal_all, 'trueFISP_2D_Radial.seq', [n],   
        N_readout=N_readout, N_spokes=N_spokes,
        Nx_img=Nx_img, Ny_img=Ny_img, fov=fov, device=device)
    ti_ms = ti_eff([n], TI_start, gre_block_dur) * 1000
    images.append(np.rot90(img, k=1 ))
    ti_effs.append(ti_ms)
    print(f'Readout {n:2d}  TI_eff={ti_ms:.0f} ms')

print(f'Done — {len(images)} images, TI_eff {ti_effs[0]:.0f} -> {ti_effs[-1]:.0f} ms')

In [ ]:
n_imgs = len(images)
ncols = min(5, n_imgs)
nrows = int(np.ceil(n_imgs / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.5, nrows * 2.5))
for i, ax in enumerate(np.atleast_1d(axes).ravel()):
    if i < n_imgs:
        ax.imshow(images[i], cmap='gray')
        ax.set_title(f'RO {readout_range.start + i}\nTI={ti_effs[i]:.0f}ms', fontsize=9)
    ax.axis('off')
plt.suptitle(f'TrueFISP — {n_imgs} readout-resolved images', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
vmax = images[0].max()  

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for p, ax in enumerate(axes.ravel()):
    ax.imshow(images[p], cmap='gray', vmin=0, vmax=vmax)
    ax.set_title(f'Frame {p}\nTI={ti_effs[p]:.0f}ms', fontsize=9)
    ax.axis('off')

In [ ]:
for p, img in enumerate(images):
    print(f'Frame {p}: max={img.max():.4g}  mean={img.mean():.4g}')

In [ ]:
dc_idx = Nx // 2
dc_signal = signal_all[:, :, dc_idx]                       
dc_mean = dc_signal.mean(dim=0).cpu().numpy()      

ti_vals = np.array([ti_eff(p, 1, TI_start, gre_block_dur) * 1000 for p in range(N_readout)])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ti_vals, dc_mean / dc_mean.max(), 'ko-', markersize=8, label='DC signal (normalized)')
ax.set_xlabel('Effective TI (ms)')
ax.set_ylabel('Normalized signal magnitude')
ax.set_title('TrueFISP magnetization recovery — DC k-space signal vs TI')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.optimize import curve_fit

def ir_truefisp_model(t, S_StSt, S0, T1_star):
    return S_StSt - (S0 + S_StSt) * np.exp(-t / T1_star)

def fit_ir_truefisp_pixel(signal_curve, t_array, alpha_deg, p0=None):

    alpha = np.deg2rad(alpha_deg)
    if p0 is None:
        p0 = [signal_curve.max(), -signal_curve.max(), 0.5]  
    try:
        popt, _ = curve_fit(ir_truefisp_model, t_array, signal_curve, p0=p0, maxfev=5000)
        S_StSt, S0, T1_star = popt
        T1 = T1_star * (S0 / S_StSt) * np.cos(alpha / 2)
        T2 = T1_star * np.sin(alpha / 2)**2 * (1 - (S_StSt / S0) * np.cos(alpha / 2))**-1
        M0 = S0 / np.sin(alpha / 2)
        return T1, T2, M0
    except Exception:
        return None, None, None


def fit_ir_truefisp_map(image_series, t_array, alpha_deg):

    N_frames, Ny, Nx = image_series.shape
    T1_map = np.zeros((Ny, Nx))
    T2_map = np.zeros((Ny, Nx))
    M0_map = np.zeros((Ny, Nx))

    for y in range(Ny):
        for x in range(Nx):
            curve = image_series[:, y, x]
            if curve.max() < 1e-6:   
                continue
            T1, T2, M0 = fit_ir_truefisp_pixel(curve, t_array, alpha_deg)
            if T1 is not None:
                T1_map[y, x] = T1
                T2_map[y, x] = T2
                M0_map[y, x] = M0

    return T1_map, T2_map, M0_map